# casmi26 v4b 段 1 の 2 シャード同時の T4 試走（Issue #88、2026-11-14 夜）

昼の試走（`taichiiiii/casmi26-v4b-sim1-trial` v1）と同じ TC 既定の引数＋`--nshard 6 --max_structs 40` で、シャード 0 を GPU 0（`CUDA_VISIBLE_DEVICES=0`）、シャード 1 を GPU 1 で同時に回し、減速倍率 s = 同時実行のシャード 0 の t_query 合計 ÷ 昼のシャード 0 の t_query 合計 を `/kaggle/working/out/trial_report.json` に書く。入力の照合は昼と同じ（1 つでも違えば段 1 を回さない）。段 1 は 45 分、ノート全体 60 分で打ち切り。生成: `scripts/train/make_sim1_trial_kernel.py --two-shards`。

In [ ]:
"""段 1（TC `simulate_v1ctx.py`）の Kaggle T4 試走の部品（Issue #88）。

標準ライブラリだけで書く。生成するノートブックにこのファイルのソースをそのまま 1 セルとして埋め込み、
同じ関数をローカルのテストでも使う。

- 入力の照合表を配布元の記録（spec の README、v4g の MANIFEST、v4b の MANIFEST）から作る
- 段 1 の標準出力（進み具合の行）から処理数と秒数を読む
- 本番の GPU 時間の外挿（式は Issue #88 の事前登録のとおり）
"""
import hashlib
import json
import re

# 段 1 の入力: work からの相対パス → (配布元のデータセット, データセットの中の相対パス, 記録の出どころ)
SPEC_FILES = ("spec_off.npy", "spec_mz.npy", "spec_it.npy", "spec_meta.parquet", "split.parquet")
POOL_FILES = ("train_structs.parquet", "pool_meta.parquet", "pool_fp.npy", "fp_bits.npy",
              "pool_frag_off.npy", "pool_frag_mass.npy", "train_fp_sel.npy")
MODEL_FILES = (("fpnet_a.pt", "fe_A.pt"), ("fpnet_b.pt", "fe_B.pt"),
               ("analysis/dreams_ft/s2/pilot_d1_step20k_ema.pt", "fe_D.pt"))
FE_V4_CODE = ("ctxlib.py", "v1engine.py", "hooks_v1.py", "derive6.py", "fragcaps.py", "chemfeat.py")
N_V1_CODE = 17

# 本番の大きさ（作者の記録。TC `fe_v4/after_sim1x_d2.log` の 15,701 クエリ、`v1fe_sim1x/log_*.txt` の 892×4＋891×2 構造）
TOTAL_QUERIES = 15701
TOTAL_STRUCTS = 5350
# 作者のシャード 0 の記録（`v1fe_sim1x/log_0.txt`）: 最初の 40 構造で 200 クエリ 641 s、6 シャード合計 36,099 シャード秒
AUTHOR_FIRST40 = dict(structs=40, queries=200, seconds=641.0)
AUTHOR_TOTAL_SHARD_SECONDS = 36099.0

_SPEC_ROW = re.compile(r"^\|\s*cache/(\S+)\s*\|\s*([\d,]+)\s*\|\s*([0-9a-f]{64})\s*\|")
_PROGRESS = re.compile(r"shard (\d+): (\d+)/(\d+) structs, (\d+) queries, (\d+)s \(([\d.]+)s/q")
_HEADER = re.compile(r"\[sim1x\] shard (\d+): (\d+) structures, (\d+) done, seed (\d+), init (\d+)s")
_DONE = re.compile(r"\[sim1x\] done shard (\d+): (\d+) new queries (\d+)s")


def parse_spec_readme(text):
    """spec cache の docs/README.md の表 → {名前: (バイト数, sha256)}。"""
    out = {}
    for line in text.splitlines():
        m = _SPEC_ROW.match(line.strip())
        if m:
            out[m.group(1)] = (int(m.group(2).replace(",", "")), m.group(3))
    return out


def expected_inputs(spec_readme_text, v4g_manifest, v4b_manifest):
    """段 1 のデータ入力の照合表。記録が無いものは sha256=None（照合せず値だけ残す）。"""
    spec = parse_spec_readme(spec_readme_text)
    out = {}
    for n in SPEC_FILES:
        b, s = spec.get(n, (None, None))
        out[n] = dict(dataset="ahmedberatozer/casmi26-spec-cache", rel=f"cache/{n}", bytes=b, sha256=s,
                      record="casmi26-spec-cache/docs/README.md" if s else "記録なし")
    gf = v4g_manifest["files"]
    for n in POOL_FILES:
        e = gf.get(f"v2_pool/{n}")
        out[n] = dict(dataset="ahmedberatozer/casmi26-v2-pool", rel=n, bytes=e["bytes"] if e else None,
                      sha256=e["sha256"] if e else None,
                      record="casmi26-attributed-v4g-reproduction-assets/MANIFEST.json" if e else "記録なし")
    bf = v4b_manifest["files"]
    for work_rel, name in MODEL_FILES:
        s = bf.get(name)
        out[work_rel] = dict(dataset="ahmedberatozer/casmi26-v4b-models", rel=name, bytes=None, sha256=s,
                             record="casmi26-v4b-models/MANIFEST.json" if s else "記録なし")
    return out


def expected_code(v4b_manifest):
    """TC の写し（README §4 の配置先）→ v4b MANIFEST の sha256。v1 casmi 17 本＋fe_v4 6 本＝23 本。"""
    bf = v4b_manifest["files"]
    out = {}
    for k, s in bf.items():
        if k.startswith("code/casmi/") and k.endswith(".py"):
            out["work/analysis/v1fe/code/casmi/" + k[len("code/casmi/"):]] = s
    for n in FE_V4_CODE:
        out["work/analysis/fe_v4/" + n] = bf["code/fe_v4/" + n]
    assert len(out) == N_V1_CODE + len(FE_V4_CODE), len(out)
    return out


def sha256_file(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            b = fh.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def compare(expected, actual):
    """expected/actual: {名前: {sha256, bytes}}。食い違いの一覧（記録が無いものは照合しない）。"""
    bad = []
    for n, e in expected.items():
        a = actual.get(n)
        if a is None:
            bad.append(dict(name=n, why="見つからない"))
            continue
        if e.get("sha256") and a.get("sha256") != e["sha256"]:
            bad.append(dict(name=n, why="sha256 が違う", expected=e["sha256"], actual=a.get("sha256")))
        elif e.get("bytes") is not None and a.get("bytes") != e["bytes"]:
            bad.append(dict(name=n, why="大きさが違う", expected=e["bytes"], actual=a.get("bytes")))
    return bad


def stage_timeout(elapsed, budget=25 * 60, cap=20 * 60, margin=120):
    """段 1 の subprocess の timeout（秒）。残り時間 − 余裕 と cap の短い方。0 以下なら回さない。"""
    return int(min(cap, budget - elapsed - margin))


def parse_sim_log(lines):
    """段 1 の標準出力 → 処理数と秒数。打ち切られても、最後の進み具合の行の値を返す。"""
    r = dict(header=None, progress=[], done=None)
    for ln in lines:
        m = _HEADER.search(ln)
        if m:
            r["header"] = dict(shard=int(m.group(1)), structs=int(m.group(2)), done_before=int(m.group(3)),
                               seed=int(m.group(4)), init_s=int(m.group(5)))
        m = _PROGRESS.search(ln)
        if m:
            r["progress"].append(dict(structs=int(m.group(2)), of=int(m.group(3)), queries=int(m.group(4)),
                                      seconds=int(m.group(5)), s_per_q=float(m.group(6))))
        m = _DONE.search(ln)
        if m:
            r["done"] = dict(queries=int(m.group(2)), seconds=int(m.group(3)))
    return r


def summarize(parsed, rows_meta_queries=None):
    """処理した構造数・クエリ数・秒数（構造の処理ループの秒。初期化は含まない）。

    完走なら done 行（構造数は見出しの行の対象数 − 処理済み）、打ち切りなら最後の進み具合の行を使う。
    """
    if parsed["done"] is not None and parsed["header"] is not None:
        n_s = parsed["header"]["structs"] - parsed["header"]["done_before"]
        n_q, sec, src = parsed["done"]["queries"], float(parsed["done"]["seconds"]), "done 行"
        if rows_meta_queries is not None:
            n_q = rows_meta_queries
    elif parsed["progress"]:
        p = parsed["progress"][-1]
        n_s, n_q, sec, src = p["structs"], p["queries"], float(p["seconds"]), "最後の進み具合の行（打ち切り）"
    else:
        return dict(structs=0, queries=0, seconds=None, s_per_struct=None, s_per_query=None, source="なし")
    return dict(structs=n_s, queries=n_q, seconds=sec, s_per_struct=sec / n_s if n_s else None,
                s_per_query=sec / n_q if n_q else None, source=src)


def extrapolate(s_per_query, s_per_struct, parallel=(1, 2, 6)):
    """本番の段 1 の GPU 時間（時間）。並行で 1 本あたりが遅くなる分は入れていない（2・6 本は楽観寄り）。

    補正: 作者の記録でも最初の 40 構造は全体平均より重い（641/200 s/q 対 36,099/15,701 s/q）ので、
    その比を掛けた値も出す（推定）。
    """
    author_first = AUTHOR_FIRST40["seconds"] / AUTHOR_FIRST40["queries"]
    author_all = AUTHOR_TOTAL_SHARD_SECONDS / TOTAL_QUERIES
    corr = author_all / author_first
    out = dict(total_queries=TOTAL_QUERIES, total_structs=TOTAL_STRUCTS, author_first40_s_per_q=round(author_first, 4),
               author_all_s_per_q=round(author_all, 4), correction=round(corr, 4), by_parallel={})
    for k in parallel:
        e = {}
        if s_per_query:
            e["by_query_h"] = round(s_per_query * TOTAL_QUERIES / k / 3600, 3)
            e["by_query_corrected_h"] = round(s_per_query * corr * TOTAL_QUERIES / k / 3600, 3)
        if s_per_struct:
            e["by_struct_h"] = round(s_per_struct * TOTAL_STRUCTS / k / 3600, 3)
        out["by_parallel"][str(k)] = e
    return out


def dumps(obj):
    return json.dumps(obj, ensure_ascii=False, indent=2, default=str)


In [ ]:
"""段 1 の 2 シャード同時の試走の部品（Issue #88、2026-11-14 夜）。

標準ライブラリだけで書く。2 シャードのノートには `sim1_trial.py` に続けてこのファイルのソースを 1 セルとして
埋め込む（1 シャードの既定のノートは変えないため、`sim1_trial.py` には手を入れない）。

- 減速倍率 s の定義（#88 の事前登録）: 同時実行のシャード 0 の t_query 合計 ÷ 昼の試走のシャード 0 の t_query 合計
- 2 本の subprocess を同じ締め切りで待ち、超えたものだけ止める
- nvidia-smi の全 GPU の行を読む
"""

# 2 シャードを同時に回す: シャード番号 → 見せる GPU（CUDA_VISIBLE_DEVICES）
SHARD_GPU = ((0, 0), (1, 1))
NSHARD = 6
MAX_STRUCTS = 40
BUDGET2_S = 60 * 60       # ノート全体
STAGE2_CAP_S = 45 * 60    # 段 1（2 本とも同じ締め切り）
STAGE2_MARGIN_S = 180


def shard_args(shard, nshard=NSHARD, max_structs=MAX_STRUCTS):
    """TC 既定に足す引数（昼の試走の `--shard 0 --nshard 6 --max_structs 40` のシャード番号だけを変える）。"""
    return ["--shard", str(shard), "--nshard", str(nshard), "--max_structs", str(max_structs)]


def shard_env(base_env, tc_env, gpu):
    env = dict(base_env)
    env.update(tc_env)
    env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    return env


def stage2_timeout(elapsed, budget=BUDGET2_S, cap=STAGE2_CAP_S, margin=STAGE2_MARGIN_S):
    """段 1 の締め切り（秒）。残り時間 − 余裕 と cap の短い方。60 以下なら回さない。"""
    return int(min(cap, budget - elapsed - margin))


def parse_gpu_query(text):
    """`nvidia-smi --query-gpu=index,memory.used,utilization.gpu --format=csv,noheader,nounits` → {index: (mem, util)}。"""
    out = {}
    for ln in (text or "").splitlines():
        parts = [x.strip() for x in ln.split(",")]
        if len(parts) != 3:
            continue
        try:
            out[int(parts[0])] = (float(parts[1]), float(parts[2]))
        except ValueError:
            continue
    return out


def update_gpu_peak(peak, sample):
    """peak: {str(index): {mem_mb, util}} を sample で更新する。"""
    for i, (mem, util) in sample.items():
        e = peak.setdefault(str(i), dict(mem_mb=0.0, util=0.0))
        e["mem_mb"] = max(e["mem_mb"], mem)
        e["util"] = max(e["util"], util)
    return peak


def wait_all(procs, deadline, now, wait_one, kill_one):
    """procs: {key: proc}。deadline（now() と同じ時計）までに終わらないものだけ kill_one で止める。

    wait_one(proc, timeout) は返り値を返すか TimeoutError 相当の例外を投げる。戻り値 {key: (rc, timed_out)}。
    """
    res = {}
    for k, p in procs.items():
        left = deadline - now()
        try:
            res[k] = (wait_one(p, max(1.0, left)), False)
        except Exception:
            res[k] = (kill_one(p), True)
    return res


def t_query_sum(values):
    vals = [float(v) for v in values]
    return dict(n=len(vals), sum=sum(vals), mean=(sum(vals) / len(vals)) if vals else None)


def slowdown(concurrent_shard0_sum, day_shard0_sum):
    """s = 同時実行のシャード 0 の t_query 合計 ÷ 昼の試走のシャード 0 の t_query 合計。"""
    if not concurrent_shard0_sum or not day_shard0_sum:
        return None
    return concurrent_shard0_sum / day_shard0_sum


def query_signature(sids, regimes, qidxs):
    """クエリの並び（sid, regime, qidx）の sha256。昼と同じクエリかを確かめるのに使う。"""
    import hashlib
    import json
    rows = [[int(s), str(r), [int(x) for x in q]] for s, r, q in zip(sids, regimes, qidxs)]
    return hashlib.sha256(json.dumps(rows, separators=(",", ":")).encode()).hexdigest()


def cpu_model(lscpu_text):
    """lscpu の出力 → (型番, CPU(s))。"""
    model, n = None, None
    for ln in (lscpu_text or "").splitlines():
        if ":" not in ln:
            continue
        k, v = [x.strip() for x in ln.split(":", 1)]
        if k == "Model name" and model is None:
            model = v
        elif k == "CPU(s)" and n is None:
            try:
                n = int(v)
            except ValueError:
                pass
    return model, n


In [ ]:
# ---- 定数（scripts/train/make_sim1_trial_kernel.py が配布元の記録から埋め込む） ----
SLUG = 'taichiiiii/casmi26-v4b-sim1-trial2'
TC_ARGS = ['--out', 'work/analysis/v1fe/sim1x/rows', '--ctx', 'work/analysis/v1fe/sim1x/ctx', '--folds', 'np,nplib,syn,plusk', '--regimes', 'c1,c2,c3', '--seed', '21', '--part_every', '40', '--engine_models', 'work/fpnet_a.pt']
TRIAL_ARGS = ['--shard', '0', '--nshard', '6', '--max_structs', '40']
TC_ENV = {'PYTHONIOENCODING': 'utf-8', 'NUMBA_NUM_THREADS': '2', 'OMP_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2', 'OPENBLAS_NUM_THREADS': '2'}
PIP = ['rdkit==2026.03.3', 'lightgbm==4.7.0']
EXP_DATA = json.loads('{"spec_off.npy": {"dataset": "ahmedberatozer/casmi26-spec-cache", "rel": "cache/spec_off.npy", "bytes": 20317000, "sha256": "af2be108a532a278ec4563c023808b7fa19e40fdf4ea7dc888e980bb6f618e6c", "record": "casmi26-spec-cache/docs/README.md"}, "spec_mz.npy": {"dataset": "ahmedberatozer/casmi26-spec-cache", "rel": "cache/spec_mz.npy", "bytes": 532912560, "sha256": "482332fec3fabcc46e06ce1b23d79e633af7009b8190a3dcd0a675121b42342b", "record": "casmi26-spec-cache/docs/README.md"}, "spec_it.npy": {"dataset": "ahmedberatozer/casmi26-spec-cache", "rel": "cache/spec_it.npy", "bytes": 532912560, "sha256": "9e127f00eb9d4217403991304e69166adc9a36a5e624ce7fd1ddbf6b1417cc23", "record": "casmi26-spec-cache/docs/README.md"}, "spec_meta.parquet": {"dataset": "ahmedberatozer/casmi26-spec-cache", "rel": "cache/spec_meta.parquet", "bytes": 54036239, "sha256": "4f968592dc75b9bf2b6256dda394be42bad3cd07fd243ff0078101e066a44376", "record": "casmi26-spec-cache/docs/README.md"}, "split.parquet": {"dataset": "ahmedberatozer/casmi26-spec-cache", "rel": "cache/split.parquet", "bytes": 9641941, "sha256": "d05a2f9f71cffd480b92581c458547f7aeb782436b07af4fbf8f696f96649656", "record": "casmi26-spec-cache/docs/README.md"}, "train_structs.parquet": {"dataset": "ahmedberatozer/casmi26-v2-pool", "rel": "train_structs.parquet", "bytes": 23306834, "sha256": "f20aa1cac23af42e430a6633707fda34e48dea97114bdc3e443897f692df3656", "record": "casmi26-attributed-v4g-reproduction-assets/MANIFEST.json"}, "pool_meta.parquet": {"dataset": "ahmedberatozer/casmi26-v2-pool", "rel": "pool_meta.parquet", "bytes": 36692358, "sha256": "82b8c40315611289b9a4004b3af18d6d3402b3cd5baa4ee5723da07b7e2f6bb0", "record": "casmi26-attributed-v4g-reproduction-assets/MANIFEST.json"}, "pool_fp.npy": {"dataset": "ahmedberatozer/casmi26-v2-pool", "rel": "pool_fp.npy", "bytes": 908986707, "sha256": "f02d91eeb4e2be5687ce678c199aa09a3acda1e821478372fc23fd1c7edf55e8", "record": "casmi26-attributed-v4g-reproduction-assets/MANIFEST.json"}, "fp_bits.npy": {"dataset": "ahmedberatozer/casmi26-v2-pool", "rel": "fp_bits.npy", "bytes": 41032, "sha256": "c3869e86449d6fed6e79e9655f8a7d35bd55c3138ecddfc1dae5072627e78677", "record": "casmi26-attributed-v4g-reproduction-assets/MANIFEST.json"}, "pool_frag_off.npy": {"dataset": "ahmedberatozer/casmi26-v2-pool", "rel": "pool_frag_off.npy", "bytes": 5685744, "sha256": "354015a6eab8b6762f9124384e959d606e961c35b999d1b088153d69538765e7", "record": "casmi26-attributed-v4g-reproduction-assets/MANIFEST.json"}, "pool_frag_mass.npy": {"dataset": "ahmedberatozer/casmi26-v2-pool", "rel": "pool_frag_mass.npy", "bytes": 357116708, "sha256": "9afbdc02856b287e06e6c9f61a9f0032b2dd53b20557942c1d00542d9c490ffc", "record": "casmi26-attributed-v4g-reproduction-assets/MANIFEST.json"}, "train_fp_sel.npy": {"dataset": "ahmedberatozer/casmi26-v2-pool", "rel": "train_fp_sel.npy", "bytes": 352761118, "sha256": "0e6ddfbfb188b6cffb9f1b0ad9c7ba268ef709ce60b2711d05f98096cd772a8d", "record": "casmi26-attributed-v4g-reproduction-assets/MANIFEST.json"}, "fpnet_a.pt": {"dataset": "ahmedberatozer/casmi26-v4b-models", "rel": "fe_A.pt", "bytes": null, "sha256": "edccf479a2a2e53461a06bc5e1aee4341b22abb32302c4f6d2c45281000d742a", "record": "casmi26-v4b-models/MANIFEST.json"}, "fpnet_b.pt": {"dataset": "ahmedberatozer/casmi26-v4b-models", "rel": "fe_B.pt", "bytes": null, "sha256": "6f60a22b3c3b21aa48d47328bb9523bdf07cc004bfb72966e0f9713507c24f2e", "record": "casmi26-v4b-models/MANIFEST.json"}, "analysis/dreams_ft/s2/pilot_d1_step20k_ema.pt": {"dataset": "ahmedberatozer/casmi26-v4b-models", "rel": "fe_D.pt", "bytes": null, "sha256": "8ae5bee2c978f4f01ea7cfe29b514cd3e9112006ab51aa8b13f20e26e9365362", "record": "casmi26-v4b-models/MANIFEST.json"}}')
EXP_CODE = json.loads('{"work/analysis/v1fe/code/casmi/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "work/analysis/v1fe/code/casmi/build.py": "86776c14e237dc64dbf965d9bc200cc670dc7226bf946938380729d7e80c56f8", "work/analysis/v1fe/code/casmi/chem.py": "99474f23d8a67040b0c0f8c70e73592a6f7f656f50103e350ca473451e2169f1", "work/analysis/v1fe/code/casmi/derive.py": "7c1e8c3afb4c17a3ae58fbc6c36fd4cfc552da97b9129c9b46cca3cdc6ee4892", "work/analysis/v1fe/code/casmi/dreams_model.py": "5c09c3d22e8d2c09678b796cb1d57a449442437847e094f49f6ed2619a73a83c", "work/analysis/v1fe/code/casmi/engine.py": "d914bf9d5eb56cfff434b5d53a4d55881b97697aa834a627d9932a4f8c650d27", "work/analysis/v1fe/code/casmi/fpnet.py": "0ea6e372d1f045a877d350234c2ccf5d93fdd98cb23bd4c22376bf7e99547608", "work/analysis/v1fe/code/casmi/fpnet6.py": "32542bb0039b2d24d9911030559f13221e630cef7df0d10c7d65f54424da2633", "work/analysis/v1fe/code/casmi/frag.py": "809f5582494cee4cf6dcace4b04699b8427e81022b78d42162cc0239a57773de", "work/analysis/v1fe/code/casmi/graphs.py": "8dcbe16bbf3b34b7a08866234eb9abf2de713d21c93c653e7156e94e9debcd55", "work/analysis/v1fe/code/casmi/library.py": "21ee24dee9f39123da518fb17802a394e6dc46b147d5dbdf276210b60185f831", "work/analysis/v1fe/code/casmi/pool.py": "279444f6c03834adbbfc0fad321e24187585d7cd5afdc363eff083dc263a0286", "work/analysis/v1fe/code/casmi/ranker.py": "a7ee491b5a3d3decb3887afe3931adb10c11964468fb204e8c27b42757424fe4", "work/analysis/v1fe/code/casmi/spectra.py": "e2481761cdfb2515e8a945751f85f6b9e885680a8e974d7ac35c78bbf279ae74", "work/analysis/v1fe/code/casmi/structs.py": "2c70df89e8f96b3e716651fe8815d0a05af3ad26e2b448e552ea53c26ede334a", "work/analysis/v1fe/code/casmi/traindata.py": "cecb329d96362f11d16b40a26f1888f092610ff6202812408c13e2c293c6a2bc", "work/analysis/v1fe/code/casmi/xenc.py": "89fbb470e6fc085f7363bf79e4517c06d43eed3c1df197b7f374b65dacdb7310", "work/analysis/fe_v4/ctxlib.py": "c050e4b0ac937a7d65fa14ac55c6904e2232cb077830a20b653ac6df22922943", "work/analysis/fe_v4/v1engine.py": "9074546910ba850a98bb95afe531669c62a26c15ab8b6bf9d6b6020c77ea0e8a", "work/analysis/fe_v4/hooks_v1.py": "d0cb1e37566a54bbb3082ccd29579ac47fc2526c5d64ef073dc0e628ee5657e8", "work/analysis/fe_v4/derive6.py": "0e4b436d6b978b2e2e1af8767d6b3c32a6b1fc0e4b64960d5eeeaa8885ddf68a", "work/analysis/fe_v4/fragcaps.py": "ba6280f9a46ca28cd72b8a7a0324724a73fd0c67a13acdd82a8193ac30d17a83", "work/analysis/fe_v4/chemfeat.py": "8fedf317042f989b1569ca65c0d6d87a0387f9fc9934504420a95d8d973f8746"}')
EMBED_SOURCES = {'casmi26-spec-cache/docs/README.md': '2c9c357bce17ec2e73a20985f8c1f925ad1a1f593664deb304880e5ada53546c', 'casmi26-attributed-v4g-reproduction-assets/MANIFEST.json': '2593c1e95b989309e9c5057d1c00216c22c75f50d87a960b363d24f99f1bad8f', 'casmi26-v4b-models/MANIFEST.json': 'e64b960037769971e146a35932e29ed3f042de1716e4dbe281ebd84a7cfc3b6a'}
DAY_SHARD0 = {'kernel': 'taichiiiii/casmi26-v4b-sim1-trial v1', 'meta_sha256': '253c729adfcc27817a027398872dd59dd3cf183f65300ec181d2734b90bcb51c', 'n_queries': 200, 'n_structs': 40, 't_query_sum': 928.8429119586945, 'query_signature': '6c64b7d4df0129725e3387e963463b0702475b1c8b4785a966449c16feca61f1', 'cpu': 'Intel(R) Xeon(R) CPU @ 2.00GHz', 'vcpus': 4}

In [ ]:
# ---- Issue #88: TC 段 1 (simulate_v1ctx.py) の 2 シャード同時の T4 試走。ノート全体 60 分・段 1 は 45 分で打ち切り ----
import os, sys, time, json, glob, shutil, signal, subprocess, threading, platform, shlex
from pathlib import Path
T0 = time.time()
BUDGET = BUDGET2_S
PY = shlex.quote(sys.executable)
OUT = Path('/kaggle/working/out'); OUT.mkdir(parents=True, exist_ok=True)
R = Path('/tmp/R')                      # 作業場所は /tmp（/kaggle/working に 4 GB の symlink を置かない）
REPORT = dict(issue=88, slug=SLUG, started_unix=T0, budget_s=BUDGET, args=TC_ARGS + TRIAL_ARGS, env=TC_ENV,
              status='started', steps=[])
log = lambda *a: print(*a, flush=True)
el = lambda: round(time.time() - T0, 1)

def save():
    REPORT['elapsed_s'] = el()
    (OUT / 'trial_report.json').write_text(dumps(REPORT))

def step(name, **kw):
    REPORT['steps'].append(dict(name=name, t=el(), **kw)); save(); log('STEP', name, el(), kw if kw else '')

def sh(cmd, timeout=60):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return (p.stdout + p.stderr).strip()
    except Exception as e:
        return f'ERR {e!r}'

def main():
    # 1. 環境
    REPORT['env_kaggle'] = dict(python=sys.version.split()[0], platform=platform.platform(),
                                nvidia_smi=sh('nvidia-smi'), lscpu=sh('lscpu'), nproc=sh('nproc'),
                                meminfo=sh('head -3 /proc/meminfo'), df=sh('df -h /tmp /kaggle/working'))
    step('env')
    pip = {}
    for spec in PIP:
        o = sh(f'{PY} -m pip install -q {spec}', timeout=300)
        pip[spec] = o[-1500:] if o else 'ok'
    REPORT['pip'] = pip
    vers = {}
    for mod, expr in (('torch', "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"),
                      ('rdkit', "from rdkit import rdBase; print(rdBase.rdkitVersion)"),
                      ('numpy', "import numpy; print(numpy.__version__)"), ('pandas', "import pandas; print(pandas.__version__)"),
                      ('pyarrow', "import pyarrow; print(pyarrow.__version__)"), ('numba', "import numba; print(numba.__version__)"),
                      ('lightgbm', "import lightgbm; print(lightgbm.__version__)")):
        vers[mod] = sh(f'{PY} -c "{expr}"', timeout=120)[-300:]
    REPORT['versions'] = vers
    REPORT['tc_env'] = 'TC README §4: python 3.14.5, torch 2.14.0+cu130, lightgbm 4.7.0, rdkit 2026.03.3（ここと違う版は versions を見る）'
    step('pip', versions=REPORT['versions'])

    # 2. 入力の場所
    inp = Path('/kaggle/input')
    def one(pattern, must=None):
        hits = [p for p in inp.rglob(pattern) if must is None or must(p)]
        assert len(hits) == 1, (pattern, [str(h) for h in hits])
        return hits[0]
    TC = one('simulate_v1ctx.py').parent.parent                 # TC の根（fe_v4/ の親）
    SPEC = one('spec_off.npy').parent                            # cache/
    POOL = one('pool_fp.npy').parent
    V4B = one('fe_D.pt').parent
    REPORT['mounts'] = dict(tc=str(TC), spec=str(SPEC), pool=str(POOL), v4b=str(V4B))
    # 配布元の記録をマウントから読み直し、ノートに埋め込んだ値と同じか
    doc = {}
    m_v4b = json.loads((V4B / 'MANIFEST.json').read_text())
    readme = (SPEC.parent / 'docs' / 'README.md').read_text()
    doc['v4b_manifest_sha256'] = sha256_file(V4B / 'MANIFEST.json')
    doc['spec_readme_sha256'] = sha256_file(SPEC.parent / 'docs' / 'README.md')
    doc['v4b_manifest_same_as_embedded'] = expected_code(m_v4b) == EXP_CODE and all(
        m_v4b['files'].get(EXP_DATA[w]['rel']) == EXP_DATA[w]['sha256'] for w, _ in MODEL_FILES)
    _sp = parse_spec_readme(readme)
    doc['spec_readme_same_as_embedded'] = all(_sp.get(n) == (EXP_DATA[n]['bytes'], EXP_DATA[n]['sha256']) for n in SPEC_FILES)
    doc['embedded_sources'] = EMBED_SOURCES
    REPORT['records'] = doc
    step('records', **{k: v for k, v in doc.items() if k.endswith('embedded')})

    # 3. README §4 の配置（コードは写す、データは symlink）
    if R.exists():
        shutil.rmtree(R)
    for src, dst in (('fe_v4', 'work/analysis/fe_v4'), ('v1fe_code_casmi', 'work/analysis/v1fe/code/casmi'),
                     ('engine_v6_casmi', 'work/analysis/engine_v6/code/casmi'), ('casmi', 'src/casmi')):
        shutil.copytree(TC / src, R / dst)
    W = R / 'work'
    srcs = {}
    for n in SPEC_FILES:
        srcs[n] = SPEC / n
    for n in POOL_FILES:
        srcs[n] = POOL / n
    for w, name in MODEL_FILES:
        srcs[w] = V4B / name
    for w, p in srcs.items():
        (W / w).parent.mkdir(parents=True, exist_ok=True)
        os.symlink(p, W / w)
    step('layout')

    # 4. 照合（データ 15 本＋コード 23 本）。1 つでも違えば段 1 を回さない
    act = {}
    for w, p in srcs.items():
        act[w] = dict(sha256=sha256_file(p), bytes=p.stat().st_size)
    act_code = {rel: dict(sha256=sha256_file(R / rel)) for rel in EXP_CODE}
    bad = compare(EXP_DATA, act) + compare({k: dict(sha256=v) for k, v in EXP_CODE.items()}, act_code)
    REPORT['inputs'] = {w: dict(EXP_DATA[w], actual_sha256=act[w]['sha256'], actual_bytes=act[w]['bytes']) for w in EXP_DATA}
    REPORT['code_checked'] = len(act_code)
    # 記録の無い TC のコード（段 1 の入口 simulate_v1ctx.py ほか）は sha256 だけを残す
    REPORT['code_unrecorded_sha256'] = {str(f.relative_to(R)): sha256_file(f) for f in sorted((R / 'work').rglob('*.py'))
                                        if str(f.relative_to(R)) not in EXP_CODE}
    REPORT['mismatch'] = bad
    step('verify', n_data=len(act), n_code=len(act_code), n_bad=len(bad))
    if bad:
        REPORT['status'] = 'stopped_sha256_mismatch'
        return

    # 5. 段 1 を 2 シャード同時に（シャード 0 → GPU 0、シャード 1 → GPU 1）
    tmo = stage2_timeout(time.time() - T0)
    REPORT['stage1_timeout_s'] = tmo
    REPORT['day_shard0'] = DAY_SHARD0
    if tmo <= 60:
        REPORT['status'] = 'skipped_no_time'
        return
    use_time = os.path.exists('/usr/bin/time') and sh('/usr/bin/time -v true').find('Maximum resident') >= 0
    procs, lines, cmds = {}, {}, {}
    t_start = time.time()
    for shard, gpu in SHARD_GPU:
        cmd = [sys.executable, 'work/analysis/fe_v4/simulate_v1ctx.py'] + TC_ARGS + shard_args(shard)
        if use_time:
            cmd = ['/usr/bin/time', '-v'] + cmd
        cmds[str(shard)] = dict(cmd=cmd, CUDA_VISIBLE_DEVICES=str(gpu))
        lines[shard] = []
        procs[shard] = subprocess.Popen(cmd, cwd=str(R), env=shard_env(os.environ, TC_ENV, gpu), stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True, start_new_session=True, bufsize=1)
    REPORT['stage1_cmds'] = cmds
    REPORT['stage1_start_gap_s'] = round(time.time() - t_start, 2)
    step('stage1_started', shards=[s for s, _ in SHARD_GPU])

    def reader(shard):
        p = procs[shard]
        with open(OUT / f'sim1_stdout_s{shard}.log', 'w') as fh:
            for ln in p.stdout:
                lines[shard].append(ln.rstrip('\n')); fh.write(ln); fh.flush(); print(f'[s{shard}] ' + ln, end='', flush=True)

    def tree_rss_mb(root):
        try:
            import psutil
            pr = psutil.Process(root)
            return sum(q.memory_info().rss for q in [pr] + pr.children(recursive=True)) / 2**20
        except ImportError:
            pass
        except Exception:
            return 0.0
        tot, todo, seen = 0, [root], set()
        while todo:
            q = todo.pop()
            if q in seen:
                continue
            seen.add(q)
            try:
                for l in open(f'/proc/{q}/status'):
                    if l.startswith('VmRSS:'):
                        tot += int(l.split()[1])
                for t in os.listdir(f'/proc/{q}/task'):
                    try:
                        todo += [int(c) for c in open(f'/proc/{q}/task/{t}/children').read().split()]
                    except OSError:
                        pass
            except OSError:
                pass
        return tot / 1024

    peak = dict(rss_mb={str(s): 0.0 for s, _ in SHARD_GPU}, rss_total_mb=0.0, gpu={}, min_mem_available_mb=None,
                loadavg1_max=0.0, n_samples=0)
    ends = {}   # シャードごとの終了を見た時刻（2 秒おきの監視。2 本が重なっていた時間を出す）
    def monitor():
        while any(p.poll() is None for p in procs.values()):
            for s, p in procs.items():
                if p.poll() is not None and s not in ends:
                    ends[s] = time.time()
            tot = 0.0
            for s, p in procs.items():
                v = tree_rss_mb(p.pid) if p.poll() is None else 0.0
                peak['rss_mb'][str(s)] = max(peak['rss_mb'][str(s)], v); tot += v
            peak['rss_total_mb'] = max(peak['rss_total_mb'], tot)
            o = sh('nvidia-smi --query-gpu=index,memory.used,utilization.gpu --format=csv,noheader,nounits', timeout=10)
            update_gpu_peak(peak['gpu'], parse_gpu_query(o))
            try:
                ma = [int(l.split()[1]) for l in open('/proc/meminfo') if l.startswith('MemAvailable:')][0] / 1024
                peak['min_mem_available_mb'] = ma if peak['min_mem_available_mb'] is None else min(peak['min_mem_available_mb'], ma)
                peak['loadavg1_max'] = max(peak['loadavg1_max'], float(open('/proc/loadavg').read().split()[0]))
            except Exception:
                pass
            peak['n_samples'] += 1
            time.sleep(2)
    ths = [threading.Thread(target=reader, args=(s,), daemon=True) for s in procs]
    for th in ths:
        th.start()
    th_m = threading.Thread(target=monitor, daemon=True); th_m.start()

    def kill_one(p):
        try:
            os.killpg(p.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            return p.wait(timeout=15)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(p.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            return p.wait()
    def wait_one(p, timeout):
        return p.wait(timeout=timeout)
    res = wait_all(procs, t_start + tmo, time.time, wait_one, kill_one)
    for th in ths:
        th.join(timeout=10)
    wall = time.time() - t_start
    for s, p in procs.items():
        ends.setdefault(s, time.time())
    REPORT['stage1_end_s'] = {str(s): round(ends[s] - t_start, 1) for s in procs}
    REPORT['stage1_overlap_s'] = round(min(ends.values()) - t_start, 1)
    st1 = {}
    for s, _ in SHARD_GPU:
        parsed = parse_sim_log(lines[s])
        gnu = None
        for ln in lines[s]:
            if 'Maximum resident set size' in ln:
                gnu = round(int(ln.split(':')[-1].strip()) / 1024, 1)
        st1[str(s)] = dict(returncode=res[s][0], timed_out=res[s][1], header=parsed['header'], done=parsed['done'],
                           progress=parsed['progress'], gnu_time_max_rss_mb=gnu, tail=lines[s][-40:], _parsed=parsed)
    REPORT['stage1'] = {k: {kk: vv for kk, vv in v.items() if kk != '_parsed'} for k, v in st1.items()}
    REPORT['stage1_wall_s'] = round(wall, 1)
    REPORT['stage1_peak'] = peak
    step('stage1', **{f'rc{k}': v['returncode'] for k, v in st1.items()}, wall_s=round(wall, 1),
         timed_out=[k for k, v in st1.items() if v['timed_out']])

    # 6. 出力の形（シャードごと）と s
    import numpy as np, pandas as pd
    sim = R / 'work/analysis/v1fe/sim1x'
    REPORT['outputs'] = {str(f.relative_to(sim)): f.stat().st_size for f in sorted(sim.rglob('*')) if f.is_file()}
    shards = {}
    for s, _ in SHARD_GPU:
        d = {}
        try:
            parts = sorted(sim.glob(f'rows_{s}_p[0-9][0-9][0-9].npz'))
            rows_q = None
            if parts:
                with np.load(parts[0], allow_pickle=True) as z:
                    X, y, g, feats = z['X'], z['y'], z['g'], list(z['features'])
                d.update(part=parts[0].name, n_parts=len(parts), n_rows=int(X.shape[0]), n_cols=int(X.shape[1]),
                         n_features=len(feats), same_as_94=bool(X.shape[1] == 94 and len(feats) == 94), dtype=str(X.dtype),
                         all_finite=bool(np.isfinite(X).all()), n_groups=int(len(np.unique(g))), n_pos=int(y.sum()))
                mp = str(parts[0]).replace('.npz', '_meta.parquet')
                m = pd.read_parquet(mp)
                rows_q = int(len(m))
                tq = t_query_sum(m.t_query)
                d['meta'] = dict(n=int(len(m)), n_structs=int(m.sid.nunique()), regime=m.regime.value_counts().to_dict(),
                                 has_truth=int(m.has_truth.sum()), t_query=tq, t_query_median=float(m.t_query.median()),
                                 t_query_max=float(m.t_query.max()),
                                 t_query_mean_by_regime={k: float(v) for k, v in m.groupby('regime').t_query.mean().items()},
                                 query_signature=query_signature(m.sid, m.regime, m.qidx), sha256=sha256_file(mp))
                shutil.copy2(mp, OUT / Path(mp).name)
            ctxs = sorted((sim / 'ctx').glob(f'ctx_s{s}_c*.pkl'))
            d['ctx'] = dict(files=len(ctxs), bytes=int(sum(c.stat().st_size for c in ctxs)))
        except Exception as e:
            d['error'] = repr(e)
        d['summary'] = summarize(st1[str(s)]['_parsed'], rows_meta_queries=rows_q)
        shards[str(s)] = d
    REPORT['shards'] = shards
    m0 = shards.get('0', {}).get('meta')
    sv = dict(definition='同時実行のシャード 0 の t_query 合計 ÷ 昼の試走のシャード 0 の t_query 合計',
              day_sum=DAY_SHARD0['t_query_sum'])
    if m0:
        sv.update(concurrent_sum=m0['t_query']['sum'], s=slowdown(m0['t_query']['sum'], DAY_SHARD0['t_query_sum']),
                  same_queries_as_day=(m0['query_signature'] == DAY_SHARD0['query_signature']),
                  n_queries=m0['n'], day_n_queries=DAY_SHARD0['n_queries'])
    model, ncpu = cpu_model(REPORT.get('env_kaggle', {}).get('lscpu', ''))
    sv['cpu_this_session'] = dict(model=model, vcpus=ncpu)
    sv['cpu_day_session'] = dict(model=DAY_SHARD0['cpu'], vcpus=DAY_SHARD0['vcpus'])
    sv['note'] = ('別のセッションなのでホストの CPU の違いが s に混ざる。numba の JIT キャッシュは 2 本で同じ /tmp/R を使う。'
                  'シャード 1 が先に終わるとシャード 0 の後半は単独で走る（stage1_end_s・stage1_overlap_s を見る）')
    n_gpu = len([l for l in REPORT.get('env_kaggle', {}).get('nvidia_smi', '').splitlines() if 'Tesla' in l or 'T4' in l])
    sv['n_gpu_lines_in_nvidia_smi'] = n_gpu
    sv['gpus_seen_by_monitor'] = sorted(peak['gpu'])
    sv['s_valid'] = bool(sv.get('s') is not None and sv.get('same_queries_as_day') and len(peak['gpu']) >= 2
                         and all(st1[str(x)]['returncode'] == 0 and not st1[str(x)]['timed_out']
                                 and (shards[str(x)]['summary'].get('structs') or 0) >= 40 for x, _ in SHARD_GPU))
    REPORT['slowdown'] = sv
    ok = all((shards[str(s)]['summary'].get('structs') or 0) >= 40 and st1[str(s)]['returncode'] == 0
             and not st1[str(s)]['timed_out'] for s, _ in SHARD_GPU)
    REPORT['status'] = 'ok' if ok else 'incomplete'
    step('summary', s=sv.get('s'), status=REPORT['status'])

try:
    main()
except Exception as e:
    import traceback
    REPORT['status'] = 'error'
    REPORT['error'] = traceback.format_exc()[-4000:]
    log(REPORT['error'])
finally:
    save()
    log('SIM1_TRIAL2', REPORT['status'], el(), 's')
    log(dumps({k: REPORT.get(k) for k in ('status', 'slowdown', 'stage1_wall_s', 'stage1_peak', 'mismatch')}))
